# Module 09: High-Performance Persistence & Parallel Computing with Joblib
## Notebook 02: Memory Mapping (`mmap`) for Massive NumPy Arrays & Zero-Copy Access

When working with large machine learning datasets or ensemble models, loading multi-gigabyte arrays into physical RAM can trigger Out-Of-Memory (OOM) crashes.
**Memory Mapping (`mmap`)** allows Python to access array data directly on disk as if it were in memory, paging data into RAM on-demand.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Understand the virtual memory mechanics of **Memory Mapping (`mmap_mode`)**.
2. Compare read-only memory mapping (`'r'`) vs. Copy-on-Write (`'c'`).
3. Load massive datasets with **zero physical RAM overhead** and instant load latencies.
4. **Advanced:** Share memory-mapped models across multiple worker processes without duplicating RAM.
5. **Advanced:** Implement **Out-of-Core Batch Evaluation** on datasets that exceed physical system memory.

In [1]:
import os
import time
import joblib
import numpy as np

print(f"Joblib Version: {joblib.__version__}")

Joblib Version: 1.6.0


### 1. Memory Mapping Mechanics: Read-Only (`'r'`) vs. Copy-on-Write (`'c'`)
When saving uncompressed NumPy arrays (`compress=0`), `joblib.dump()` writes raw contiguous binary buffers to disk.
When loading with `joblib.load(filename, mmap_mode=...)`:
- **`mmap_mode='r'`:** Read-only mode. Accessing array elements reads directly from the OS page cache. Attempting to write raises a `ValueError: assignment destination is read-only`.
- **`mmap_mode='c'`:** Copy-on-Write mode. Reading accesses disk zero-copy. Any in-place modifications write to local private RAM pages without altering the file on disk!

In [2]:
# Create a substantial test array (5,000,000 float64 elements ~ 40 MB)
array_file = "large_matrix.joblib"
original_matrix = np.linspace(0.0, 100.0, 5_000_000, dtype=np.float64)

# MUST save with compress=0 for memory mapping to work!
joblib.dump(original_matrix, array_file, compress=0)
print(f"Saved {original_matrix.nbytes / (1024**2):.1f} MB array to {array_file}")

# 1. Standard loading (loads entire 40 MB into physical RAM)
t0 = time.time()
normal_array = joblib.load(array_file)
normal_latency = (time.time() - t0) * 1000

# 2. Memory-Mapped loading (instant virtual mapping, zero data copied)
t0 = time.time()
mmap_array = joblib.load(array_file, mmap_mode="r")
mmap_latency = (time.time() - t0) * 1000

print(f"Standard Load Latency:      {normal_latency:.2f} ms")
print(f"Memory-Mapped Load Latency: {mmap_latency:.2f} ms")
print(f"Speedup: {normal_latency / max(mmap_latency, 1e-5):.1f}x faster load time!")

# Verify slice access
print(f"Direct slice access: mmap_array[1000:1005] = {mmap_array[1000:1005]}")

Saved 38.1 MB array to large_matrix.joblib
Standard Load Latency:      45.39 ms
Memory-Mapped Load Latency: 0.39 ms
Speedup: 117.2x faster load time!
Direct slice access: mmap_array[1000:1005] = [0.02    0.02002 0.02004 0.02006 0.02008]


### 2. Copy-on-Write (`mmap_mode='c'`) In Action
Copy-on-write allows worker processes to apply local transformations (e.g. centering, scaling, normalization) without corrupting the shared file on disk or requiring a full copy of the entire dataset.

In [3]:
# Load in Copy-on-Write mode
cow_array = joblib.load(array_file, mmap_mode="c")

# Modify first element in memory
cow_array[0] = 9999.99
print(f"Modified in-memory value: {cow_array[0]}")

# Verify file on disk is UNTOUCHED
check_array = joblib.load(array_file, mmap_mode="r")
print(f"Value in underlying disk file: {check_array[0]} (Original unmutated!)")

Modified in-memory value: 9999.99
Value in underlying disk file: 0.0 (Original unmutated!)


### 3. Complex Application: Out-of-Core Processing of Massive Datasets
When datasets exceed RAM:
1. Memory-map the dataset on disk (`mmap_mode='r'`).
2. Stream chunks through a processing pipeline.
3. Only the current chunk occupies physical RAM; previously processed chunks are automatically evicted by the OS kernel page cache.

In [4]:
# Stream through 5,000,000 elements in chunks of 500,000
chunk_size = 500_000
total_elements = len(mmap_array)
running_sum = 0.0

print(f"Streaming {total_elements:,} elements in chunks of {chunk_size:,}...")
for start in range(0, total_elements, chunk_size):
    end = min(start + chunk_size, total_elements)
    chunk = mmap_array[start:end] # Reads only this window into memory
    running_sum += np.sum(chunk)

computed_mean = running_sum / total_elements
true_mean = np.mean(original_matrix)
print(f"Out-of-Core Computed Mean: {computed_mean:.4f} (True: {true_mean:.4f})")

# Clean up
del mmap_array
del cow_array
del check_array
if os.path.exists(array_file):
    os.remove(array_file)

Streaming 5,000,000 elements in chunks of 500,000...
Out-of-Core Computed Mean: 50.0000 (True: 50.0000)
